# Coordinate Frame Transforms

Satellite astrodynamics requires working in several coordinate reference frames. The three most common are:

- **GCRF** (Geocentric Celestial Reference Frame) -- An Earth-centered inertial frame aligned with the J2000 equinox and equator. Equations of motion are simplest in this frame because it does not rotate with the Earth. Used for orbit propagation, conjunction analysis, and any dynamics computation.

- **ITRF** (International Terrestrial Reference Frame) -- An Earth-centered, Earth-fixed frame that rotates with the Earth. Ground station positions, GPS coordinates, and geodetic quantities (latitude, longitude, altitude) are expressed in this frame.

- **TEME** (True Equator, Mean Equinox) -- A quasi-inertial frame that is the native output of the SGP4 orbit propagator used with Two-Line Element sets (TLEs). It accounts for precession and a simplified nutation model but not the full IAU reduction. Positions from SGP4 must be rotated out of TEME before they can be compared with GCRF or ITRF data.

All frame rotations in satkit are represented as unit quaternions. Applying a rotation to a 3-vector is done with the `*` operator: `v_out = q * v_in`.

This tutorial demonstrates the rotation functions in `satkit.frametransform`.

In [ ]:
import satkit as sk
import numpy as np
import matplotlib.pyplot as plt

## Basic Frame Rotations

The `satkit.frametransform` module provides quaternion rotation functions between the major frames. Each function takes a time (or array of times) and returns the corresponding quaternion(s).

| Function | Rotation |
|---|---|
| `qitrf2gcrf(t)` | ITRF -> GCRF (full IAU-2006) |
| `qgcrf2itrf(t)` | GCRF -> ITRF (full IAU-2006) |
| `qitrf2gcrf_approx(t)` | ITRF -> GCRF (approx., ~1 arcsec) |
| `qgcrf2itrf_approx(t)` | GCRF -> ITRF (approx., ~1 arcsec) |
| `qteme2gcrf(t)` | TEME -> GCRF |
| `qteme2itrf(t)` | TEME -> ITRF |

Let's start with a concrete example: rotating a ground station's ITRF position into the GCRF inertial frame.

In [ ]:
# Define a ground station using geodetic coordinates
# MIT Haystack Observatory, Westford, MA
haystack = sk.itrfcoord(latitude_deg=42.6233, longitude_deg=-71.4882, altitude=116)
print(f'Haystack ITRF position: {haystack}')
print(f'ITRF Cartesian vector (m): {haystack.vector}')

# Pick a specific time
t = sk.time(2024, 6, 15, 12, 0, 0)

# Get the ITRF -> GCRF rotation quaternion at this time
q = sk.frametransform.qitrf2gcrf(t)

# Rotate the ITRF position vector to GCRF
pos_gcrf = q * haystack.vector
print(f'\nGCRF position at {t}: {pos_gcrf}')
print(f'Position magnitude: {np.linalg.norm(pos_gcrf):.1f} m (unchanged by rotation)')

Note that the magnitude of the position vector is preserved -- quaternion rotations are rigid-body rotations.

The inverse rotation (GCRF back to ITRF) should recover the original vector:

In [ ]:
# Rotate back from GCRF to ITRF
q_inv = sk.frametransform.qgcrf2itrf(t)
pos_roundtrip = q_inv * pos_gcrf

print(f'Original ITRF vector:    {haystack.vector}')
print(f'Round-trip ITRF vector:   {pos_roundtrip}')
print(f'Difference (m):          {np.linalg.norm(pos_roundtrip - haystack.vector):.2e}')

## Approximate vs Full IAU-2006 Reduction

The full IAU-2006/2010 ITRF-GCRF rotation accounts for precession, nutation, Earth rotation angle, and polar motion using the complete IERS conventions. This is computationally expensive.

The `_approx` variants use a simplified model that is accurate to approximately 1 arcsecond -- more than sufficient for many applications. Let's compare them.

In [ ]:
# Compare full and approximate ITRF -> GCRF rotation
t = sk.time(2024, 6, 15, 12, 0, 0)

q_full = sk.frametransform.qitrf2gcrf(t)
q_approx = sk.frametransform.qitrf2gcrf_approx(t)

# Rotate the ground station position with both
pos_full = q_full * haystack.vector
pos_approx = q_approx * haystack.vector

# Position difference in meters
pos_diff = np.linalg.norm(pos_full - pos_approx)
print(f'Position difference: {pos_diff:.3f} m')

# Convert to angular difference
# The angular error (in radians) can be estimated from the cross product of the
# unit vectors, or equivalently from the position difference at the Earth's surface
R_earth = 6371e3  # mean Earth radius, meters
angular_diff_arcsec = np.degrees(pos_diff / R_earth) * 3600
print(f'Angular difference:  {angular_diff_arcsec:.4f} arcsec')

# Compute the angle directly from the quaternion difference
# q_diff = q_full * q_approx.conjugate represents the rotation between the two
q_diff = q_full * q_approx.conjugate
angle_arcsec = np.degrees(q_diff.angle) * 3600
print(f'Quaternion angle:    {angle_arcsec:.4f} arcsec')

The difference is well under 1 arcsecond, confirming the documentation. For applications where sub-arcsecond accuracy is not needed (e.g., ground track plotting, pass prediction), the approximate version is a good choice. For high-precision orbit determination, use the full reduction.

## Time Dependence: Earth Rotation Over 24 Hours

The ITRF-to-GCRF rotation changes continuously as the Earth rotates. The dominant component is the Earth Rotation Angle (ERA), which completes roughly one full cycle per sidereal day (~23h 56m).

Let's visualize this by tracking how a ground station's GCRF position traces out a circle over 24 hours, and plot the Earth Rotation Angle directly.

In [ ]:
# Create a time array spanning 24 hours at 1-minute intervals
t0 = sk.time(2024, 6, 15, 0, 0, 0)
minutes = np.arange(0, 24 * 60 + 1)
times = np.array([t0 + sk.duration.from_minutes(m) for m in minutes])

# Get the Earth Rotation Angle at each time
era = sk.frametransform.earth_rotation_angle(times)

# Plot ERA over 24 hours
hours = minutes / 60.0
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hours, np.degrees(era) % 360)
ax.set_xlabel('Hours since midnight UTC')
ax.set_ylabel('Earth Rotation Angle (deg)')
ax.set_title('Earth Rotation Angle Over 24 Hours')
ax.set_xlim(0, 24)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Rotate the ground station position to GCRF at each time step
# and show how the inertial X-Y coordinates trace out a circle
q_arr = sk.frametransform.qitrf2gcrf(times)
pos_gcrf_arr = np.array([q * haystack.vector for q in q_arr])

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(pos_gcrf_arr[:, 0] / 1e6, pos_gcrf_arr[:, 1] / 1e6, linewidth=0.5)
ax.plot(pos_gcrf_arr[0, 0] / 1e6, pos_gcrf_arr[0, 1] / 1e6, 'go', markersize=8, label='Start (00:00 UTC)')
ax.plot(pos_gcrf_arr[-1, 0] / 1e6, pos_gcrf_arr[-1, 1] / 1e6, 'rs', markersize=8, label='End (24:00 UTC)')
ax.set_xlabel('GCRF X (thousands of km)')
ax.set_ylabel('GCRF Y (thousands of km)')
ax.set_title('Haystack Observatory GCRF Position Over 24 Hours')
ax.set_aspect('equal')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

The start and end points do not quite overlap because 24 solar hours is slightly longer than one sidereal day. The Earth completes just over one full rotation in 24 hours of UTC.

## SGP4 Output Frame: TEME to GCRF Pipeline

The SGP4 propagator outputs position and velocity in the TEME frame, which is specific to the SGP4/TLE mathematical model. To use SGP4 results alongside data in standard frames, they must be rotated.

The typical pipeline is:
- **TEME -> GCRF**: Use `qteme2gcrf` for inertial comparisons or further orbit analysis
- **TEME -> ITRF**: Use `qteme2itrf` to compute ground tracks, visibility, or geodetic sub-satellite points

Below, we propagate the ISS using SGP4 and convert the results to both GCRF and ITRF.

In [ ]:
# ISS TLE (example epoch)
line1 = '1 25544U 98067A   24167.50000000  .00016717  00000-0  10270-3 0  9002'
line2 = '2 25544  51.6400 200.0000 0001000  90.0000 270.0000 15.49000000400000'
tle = sk.TLE.from_lines([line1, line2])
print(f'TLE epoch: {tle.epoch}')

# Propagate at a single time near epoch
t = tle.epoch
pos_teme, vel_teme = sk.sgp4(tle, t)
print(f'\nTEME position (m): {pos_teme}')
print(f'TEME velocity (m/s): {vel_teme}')


In [ ]:
# Rotate TEME position and velocity to GCRF
q_teme2gcrf = sk.frametransform.qteme2gcrf(t)
pos_gcrf = q_teme2gcrf * pos_teme
vel_gcrf = q_teme2gcrf * vel_teme
print(f'GCRF position (m):   {pos_gcrf}')
print(f'GCRF velocity (m/s): {vel_gcrf}')

# Rotate TEME position to ITRF and get geodetic coordinates
q_teme2itrf = sk.frametransform.qteme2itrf(t)
pos_itrf = q_teme2itrf * pos_teme
subsatpoint = sk.itrfcoord(pos_itrf)
print(f'\nITRF position (m):   {pos_itrf}')
print(f'Sub-satellite point: {subsatpoint}')

In [ ]:
# Propagate over one full orbit and plot the ground track
period_minutes = 1440 / tle.mean_motion  # period in minutes
prop_minutes = np.linspace(0, period_minutes, 300)
prop_times = [tle.epoch + sk.duration.from_minutes(float(m)) for m in prop_minutes]

# Convert to geodetic coordinates via TEME -> ITRF
lats = []
lons = []
for t in prop_times:
    pos_teme, _ = sk.sgp4(tle, t)
    q = sk.frametransform.qteme2itrf(t)
    pos_itrf = q * pos_teme
    coord = sk.itrfcoord(pos_itrf)
    lats.append(coord.geodetic.latitude_deg)
    lons.append(coord.geodetic.longitude_deg)

# Plot ground track
fig, ax = plt.subplots(figsize=(12, 6))
ax.scatter(lons, lats, c=prop_minutes, cmap='viridis', s=2)
ax.set_xlabel('Longitude (deg)')
ax.set_ylabel('Latitude (deg)')
ax.set_title('ISS Ground Track (One Orbit, TEME -> ITRF)')
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Summary

The `satkit.frametransform` module provides all the rotations needed to convert between the standard coordinate frames used in satellite astrodynamics:

- Use `qitrf2gcrf` / `qgcrf2itrf` for converting between Earth-fixed and inertial frames (full IAU-2006 accuracy)
- Use the `_approx` variants when sub-arcsecond accuracy is not required
- Use `qteme2gcrf` and `qteme2itrf` to convert SGP4/TLE outputs to standard frames
- All functions accept both scalar times and arrays, returning quaternion(s) accordingly
- Apply rotations to 3-vectors with the `*` operator: `v_out = q * v_in`